# 🔍 NB-05: Sentence-Transformers Fine-tuning (Semantic Search)
**Goal:** Fine-tune a sentence encoder so that user questions and Claude answers have high similarity.

**Modality:** Embeddings | **Model:** sentence-transformers/all-MiniLM-L6-v2

In [ ]:
!pip install sentence-transformers datasets -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import random

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Positive pairs: (user_query, response)  label=1.0
# Negative pairs: random mismatch         label=0.0
train_examples = []
for i, d in enumerate(data):
    # positive
    train_examples.append(InputExample(texts=[d["user"], d["response"][:512]], label=1.0))
    # negative
    j = (i + random.randint(1, len(data)-1)) % len(data)
    train_examples.append(InputExample(texts=[d["user"], data[j]["response"][:512]], label=0.0))

loader = DataLoader(train_examples, shuffle=True, batch_size=16)
loss = losses.CosineSimilarityLoss(model)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=3,
    warmup_steps=30,
    output_path="./claude-sentence-encoder",
    show_progress_bar=True
)
print("✅ Semantic encoder fine-tuned!")


In [ ]:
# --- Semantic Search Demo ---
from sentence_transformers import SentenceTransformer, util
enc = SentenceTransformer("./claude-sentence-encoder")
corpus = [d["user"] for d in data]
corpus_emb = enc.encode(corpus, convert_to_tensor=True)
query = "How does backpropagation work?"
q_emb = enc.encode(query, convert_to_tensor=True)
hits = util.semantic_search(q_emb, corpus_emb, top_k=3)[0]
for h in hits:
    print(f"Score {h['score']:.3f}: {corpus[h['corpus_id']][:80]}")
